# Audio model examples

Explore text-to-music with MusicGen, text-to-speech with Bark, speech recognition with Whisper, and audio–text similarity with CLAP. Install the imported dependencies and check each model card for its license. Model weights download on first use; several examples require substantial memory and can take a long time on a CPU.

Provide `my_audio.wav` for the recognition and similarity examples. Saved outputs, where present, come from earlier runs. A CLAP softmax compares the listed prompts and should not be treated as a calibrated probability.

**Data:** No fixed dataset. Prompts are in the notebook; speech recognition and similarity examples require a local `my_audio.wav`.

## Installation

In [ ]:
!pip install transformers accelerate soundfile librosa scipy sentencepiece --quiet

## Shared imports and utilities

In [3]:
import torch
import numpy as np
import soundfile as sf
import librosa
import IPython.display as ipd
import os

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE     = torch.float16 if DEVICE == "cuda" else torch.float32
TARGET_SR = 16000

os.makedirs("outputs", exist_ok=True)

def save_and_play(audio: np.ndarray, sample_rate: int, filename: str):
    path = f"outputs/{filename}"
    sf.write(path, audio, sample_rate)
    print(f"Сохранено: {path} | sr={sample_rate} | duration={len(audio)/sample_rate:.2f}s")
    ipd.display(ipd.Audio(audio, rate=sample_rate))

def load_audio(path: str, sr: int = TARGET_SR) -> np.ndarray:
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio

def free_gpu(*names):
    """Release notebook variables by name before clearing the device cache."""
    import gc
    for name in names:
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print(f"Устройство: {DEVICE} | dtype: {DTYPE}")

Устройство: cpu | dtype: torch.float32


## Text to music with MusicGen

The examples use `facebook/musicgen-small`; larger checkpoints need more memory. The model generates EnCodec audio tokens from text.

In [4]:
from transformers import MusicgenForConditionalGeneration, AutoProcessor

MUSICGEN_MODEL = "facebook/musicgen-small"  # замените на medium/large при наличии VRAM

musicgen_processor = AutoProcessor.from_pretrained(MUSICGEN_MODEL)
musicgen_model = MusicgenForConditionalGeneration.from_pretrained(
    MUSICGEN_MODEL,
    torch_dtype=DTYPE,
).to(DEVICE)

MUSICGEN_SR = musicgen_model.config.audio_encoder.sampling_rate  # 32000
print(f"Параметров: {musicgen_model.num_parameters():,} | SR: {MUSICGEN_SR}")

/opt/miniconda3/envs/Lesson_44/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cancellation requested; stopping current tasks.


In [ ]:
# ── Text-to-Music: несколько промптов ─────────────────────────────────────────
prompts = [
    "80s pop track with a funky bass line and catchy electric guitar riff",
    "Calm piano jazz in a late-night bar, slow tempo, melancholic mood",
    "Epic orchestral cinematic score, rising tension, full strings and brass",
]

# max_new_tokens: ~50 токенов ≈ 1 сек. 500 ≈ 10 сек, 1500 ≈ 30 сек
DURATION_TOKENS = 500

for i, prompt in enumerate(prompts):
    inputs = musicgen_processor(
        text=[prompt],
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    with torch.inference_mode():
        audio_tokens = musicgen_model.generate(
            **inputs,
            max_new_tokens = DURATION_TOKENS,
            do_sample      = True,
            guidance_scale = 3.0,   # 1 = без гайденса, 3-5 оптимально
            temperature    = 1.0,   # выше = разнообразнее
        )

    # audio_tokens: [batch, channels, samples]
    audio = audio_tokens[0, 0].cpu().float().numpy()
    print(f"\n[{i+1}] {prompt}")
    save_and_play(audio, sample_rate=MUSICGEN_SR, filename=f"musicgen_{i+1}.wav")

In [ ]:
# ── Melody Conditioning (только facebook/musicgen-melody) ─────────────────────
# Раскомментируйте блок и замените MUSICGEN_MODEL выше на facebook/musicgen-melody

# MELODY_PATH = "my_melody.wav"
# melody_audio, melody_sr = sf.read(MELODY_PATH)
#
# inputs_melody = musicgen_processor(
#     text=["Upbeat electronic dance music following the given melody"],
#     audio=melody_audio,
#     sampling_rate=melody_sr,
#     return_tensors="pt",
#     padding=True,
# ).to(DEVICE)
#
# with torch.inference_mode():
#     out = musicgen_model.generate(**inputs_melody, max_new_tokens=500)
#
# audio = out[0, 0].cpu().float().numpy()
# save_and_play(audio, sample_rate=MUSICGEN_SR, filename="musicgen_melody_cond.wav")

print("Раскомментируйте блок при использовании facebook/musicgen-melody")

In [ ]:
free_gpu("musicgen_model", "musicgen_processor", "inputs", "audio_tokens")

## Text to speech with Bark

Bark supports voice presets and expressive markers such as `[laughter]` and `[sighs]`. The examples use `suno/bark-small`.

In [ ]:
from transformers import BarkModel, BarkProcessor

BARK_MODEL = "suno/bark-small"  # или suno/bark для полного качества

bark_processor = BarkProcessor.from_pretrained(BARK_MODEL)
bark_model = BarkModel.from_pretrained(
    BARK_MODEL,
    torch_dtype=DTYPE,
).to(DEVICE)

BARK_SR = bark_model.generation_config.sample_rate  # 24000
print(f"Bark SR: {BARK_SR}")

In [ ]:
# ── Примеры с разными голосами и маркерами ────────────────────────────────────
bark_examples = [
    {
        "text": "Hello! My name is Alex. [clears throat] Let me tell you something... REALLY important.",
        "voice": "v2/en_speaker_3",
        "filename": "bark_en_speaker3.wav",
    },
    {
        "text": "Привет! Это тестовая фраза на русском языке. Как дела?",
        "voice": "v2/ru_speaker_0",
        "filename": "bark_ru_speaker0.wav",
    },
    {
        "text": "[laughter] Oh wow, that is hilarious! [laughs] I can not believe it...",
        "voice": "v2/en_speaker_6",
        "filename": "bark_laugh.wav",
    },
]

for ex in bark_examples:
    inputs = bark_processor(
        ex["text"],
        voice_preset=ex["voice"],
        return_tensors="pt",
    ).to(DEVICE)

    with torch.inference_mode():
        speech = bark_model.generate(**inputs)

    audio = speech[0].cpu().float().numpy()
    print(f"\nГолос: {ex['voice']} | Текст: {ex['text']}")
    save_and_play(audio, sample_rate=BARK_SR, filename=ex["filename"])

In [ ]:
# ── Сравнение голосов на одном тексте ────────────────────────────────────────
text_for_comparison = "The quick brown fox jumps over the lazy dog."

for speaker_id in range(4):   # голоса 0-3
    voice = f"v2/en_speaker_{speaker_id}"
    inputs = bark_processor(text_for_comparison, voice_preset=voice, return_tensors="pt").to(DEVICE)

    with torch.inference_mode():
        speech = bark_model.generate(**inputs)

    audio = speech[0].cpu().float().numpy()
    print(f"\n--- {voice} ---")
    save_and_play(audio, sample_rate=BARK_SR, filename=f"bark_compare_speaker{speaker_id}.wav")

In [ ]:
# Все доступные языки/голоса:
# EN: v2/en_speaker_0..9    RU: v2/ru_speaker_0..9
# DE: v2/de_speaker_0..9    FR: v2/fr_speaker_0..9
# ES: v2/es_speaker_0..9    ZH: v2/zh_speaker_0..9
# JA: v2/ja_speaker_0..9    KO: v2/ko_speaker_0..9
# Также: hi, pl, pt, tr, ar, ...

free_gpu("bark_model", "bark_processor", "inputs", "speech")

## Speech recognition with Whisper

Whisper can transcribe speech, translate to English, detect language, and generate timestamps. This notebook uses `openai/whisper-large-v3`; smaller checkpoints need less memory.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline

WHISPER_MODEL = "openai/whisper-large-v3"  # или whisper-small/medium для экономии VRAM
WHISPER_SR    = 16000  # Whisper всегда ожидает 16kHz

whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL)
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_MODEL,
    torch_dtype=DTYPE,
).to(DEVICE)

print("Whisper загружен")

In [ ]:
# ── Базовая транскрибация ─────────────────────────────────────────────────────
AUDIO_PATH = "my_audio.wav"   # <-- ваш файл со речью

audio = load_audio(AUDIO_PATH, sr=WHISPER_SR)
print(f"Загружено: {len(audio)/WHISPER_SR:.1f}s")
ipd.display(ipd.Audio(audio, rate=WHISPER_SR))

inputs = whisper_processor(
    audio,
    sampling_rate=WHISPER_SR,
    return_tensors="pt",
).to(DEVICE, dtype=DTYPE)

with torch.inference_mode():
    predicted_ids = whisper_model.generate(
        inputs.input_features,
        # language="ru",      # раскомментировать для принудительного языка
        # task="translate",   # раскомментировать для перевода на английский
    )

transcription = whisper_processor.batch_decode(
    predicted_ids, skip_special_tokens=True
)[0]

print(f"\nТранскрипция:\n{transcription}")

In [ ]:
# ── Pipeline с временными метками (для длинных аудио) ────────────────────────
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    return_timestamps=True,   # True = сегменты | "word" = по словам
    chunk_length_s=30,        # размер чанка
    stride_length_s=5,        # перекрытие между чанками
    device=DEVICE,
)

result = asr_pipeline(AUDIO_PATH)

print(f"Полный текст:\n{result['text']}\n")
print("Сегменты с метками:")
for chunk in result.get("chunks", []):
    start, end = chunk["timestamp"]
    print(f"  [{start:.1f}s – {end:.1f}s]  {chunk['text']}")

In [ ]:
# Detect the language of the first 30 seconds.
inputs_detect = whisper_processor(
    audio[:WHISPER_SR * 30], sampling_rate=WHISPER_SR, return_tensors="pt"
).to(DEVICE, dtype=DTYPE)
with torch.inference_mode():
    language_id = whisper_model.detect_language(inputs_detect.input_features)[0].item()
language_token = whisper_processor.tokenizer.convert_ids_to_tokens(language_id)
print(f"Detected language token: {language_token}")

In [ ]:
free_gpu("asr_pipeline", "whisper_model", "whisper_processor", "inputs", "inputs_detect")

## Audio–text similarity with CLAP

CLAP embeds audio and text into a shared space. The examples compare descriptions, rank recordings and calculate cosine similarity.

In [ ]:
from transformers import ClapModel, ClapProcessor

CLAP_MODEL = "laion/larger_clap_music_and_speech"
CLAP_SR    = 48000  # CLAP ожидает 48kHz

clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL)
clap_model = ClapModel.from_pretrained(
    CLAP_MODEL,
    torch_dtype=DTYPE,
).to(DEVICE)

print("CLAP загружен")

In [ ]:
# ── Вспомогательные функции ───────────────────────────────────────────────────
def get_audio_embedding(audio_path: str) -> torch.Tensor:
    audio, _ = librosa.load(audio_path, sr=CLAP_SR, mono=True)
    inputs = clap_processor(
        audios=[audio], sampling_rate=CLAP_SR,
        return_tensors="pt", padding=True,
    ).to(DEVICE)
    # Оставляем только audio-ключи
    audio_inputs = {k: v for k, v in inputs.items() if "input_features" in k or "is_longer" in k}
    with torch.inference_mode():
        emb = clap_model.get_audio_features(**audio_inputs)
    return emb.squeeze(0).cpu().float()

def get_text_embedding(text: str) -> torch.Tensor:
    inputs = clap_processor(text=[text], return_tensors="pt", padding=True).to(DEVICE)
    text_inputs = {k: v for k, v in inputs.items() if "input_ids" in k or "attention_mask" in k}
    with torch.inference_mode():
        emb = clap_model.get_text_features(**text_inputs)
    return emb.squeeze(0).cpu().float()

def cosine_sim(a: torch.Tensor, b: torch.Tensor) -> float:
    return torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print("Функции готовы")

In [ ]:
# ── Пример 1: Zero-shot классификация аудио ───────────────────────────────────
AUDIO_PATH = "my_audio.wav"   # <-- ваш файл

candidate_labels = [
    "a dog barking",
    "rain and thunder",
    "a person speaking",
    "piano music",
    "crowd cheering",
]

audio_48k, _ = librosa.load(AUDIO_PATH, sr=CLAP_SR, mono=True)

inputs = clap_processor(
    text=candidate_labels,
    audios=[audio_48k],
    return_tensors="pt",
    padding=True,
    sampling_rate=CLAP_SR,
).to(DEVICE)

with torch.inference_mode():
    outputs = clap_model(**inputs)

# logits_per_audio: [n_audio, n_texts]
scores = outputs.logits_per_audio[0].softmax(dim=-1).cpu().float().numpy()
score_dict = dict(zip(candidate_labels, scores))

print("Zero-shot классификация:")
for label, score in sorted(score_dict.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 40)
    print(f"  {score:.3f} {bar}  {label}")

best = max(score_dict, key=score_dict.get)
print(f"\nПредсказание: '{best}' ({score_dict[best]:.3f})")

In [ ]:
# ── Пример 2: Ранжирование аудиофайлов по текстовому запросу ─────────────────
# Полезно для выбора лучшего результата из нескольких генераций

query = "relaxing jazz piano music"  # <-- ваш запрос

audio_files = [
    f"outputs/{f}" for f in os.listdir("outputs") if f.endswith(".wav")
]

if not audio_files:
    print("Нет WAV файлов в outputs/ — сначала запустите задачи 1-2")
else:
    text_emb = get_text_embedding(query)
    results = []
    for path in audio_files:
        try:
            audio_emb = get_audio_embedding(path)
            sim = cosine_sim(audio_emb, text_emb)
            results.append((path, sim))
        except Exception as e:
            print(f"Пропуск {path}: {e}")

    results.sort(key=lambda x: -x[1])

    print(f"Запрос: '{query}'\n")
    print("Ранжирование файлов:")
    for rank, (path, score) in enumerate(results, 1):
        bar = "█" * int((score + 1) * 20)  # cosine [-1,1] -> [0,40]
        print(f"  {rank}. {score:+.4f} {bar}  {os.path.basename(path)}")

In [ ]:
# ── Пример 3: Cosine similarity одного файла с набором описаний ───────────────
if audio_files:
    sample_path = audio_files[0]
    audio_emb = get_audio_embedding(sample_path)

    queries = [
        "jazz piano",
        "heavy metal guitar",
        "nature sounds with birds",
        "human speech in a room",
        "electronic dance music",
    ]

    print(f"Аудио: {os.path.basename(sample_path)}\n")
    for q in queries:
        text_emb = get_text_embedding(q)
        sim = cosine_sim(audio_emb, text_emb)
        bar = "█" * int((sim + 1) * 15)
        print(f"  {sim:+.4f} {bar}  '{q}'")

In [ ]:
free_gpu("clap_model", "clap_processor", "inputs", "outputs")

## Further reading

- [MusicGen documentation](https://huggingface.co/docs/transformers/model_doc/musicgen)
- [Bark documentation](https://huggingface.co/docs/transformers/model_doc/bark)
- [Whisper documentation](https://huggingface.co/docs/transformers/model_doc/whisper)
- [CLAP documentation](https://huggingface.co/docs/transformers/model_doc/clap)

Memory usage and generation time depend on the checkpoint, duration, precision and hardware.